In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import pickle

columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes',
'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
'num_shells','num_access_files','num_outbound_cmds','is_host_login',
'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

df = pd.read_csv("KDDTrain+.txt", names=columns)
df.drop(['difficulty'], axis=1, inplace=True)

# ---- ATTACK TYPE MAPPING ----
def map_attack(label):
    if label == "normal":
        return "normal"
    elif label in ["neptune","smurf","pod","teardrop","land","back"]:
        return "DoS"
    elif label in ["satan","ipsweep","portsweep","nmap"]:
        return "Probe"
    elif label in ["guess_passwd","ftp_write","imap","phf","multihop","warezmaster"]:
        return "R2L"
    else:
        return "U2R"

df['label'] = df['label'].apply(map_attack)

# ---- ENCODE CATEGORICAL ----
cat_cols = ['protocol_type','service','flag']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

pickle.dump(encoders, open("model/encoders.pkl","wb"))

# ---- FEATURES ----
X = df.drop(['label'], axis=1)
y = df['label']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

pickle.dump(label_encoder, open("model/label_encoder.pkl","wb"))

# ---- TRAIN ----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

pickle.dump(model, open("model/model.pkl","wb"))

print("MODEL READY")

Accuracy: 0.9987695971422902
MODEL READY


In [9]:
import pandas as pd

columns = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes',
'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
'num_shells','num_access_files','num_outbound_cmds','is_host_login',
'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

df = pd.read_csv("KDDTrain+.txt", names=columns)

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.countplot(x='label', data=df)
plt.title("Attack Type Distribution")

plt.savefig("static/plots/attack_distribution.png")
plt.close()